In [2]:
import os
import re
import shutil
import splitfolders
BASE_DIR = os.getcwd()

In [ ]:
print("Starting stratified split...")

# splitfolders automatically stratifies by iterating through each class directory
splitfolders.ratio(
    input="data_raw", 
    output="dataset",    # It will auto-create this folder
    seed=42,             # Locks the random seed so you get the same split every time
    ratio=(0.8, 0.2),    # 80% Train, 20% Validation
    group_prefix=None, 
    move=True            # False means it copies the files, keeping data_raw untouched
)

print("Split complete! Check the new 'dataset' folder.")


Starting stratified split...
Split complete! Check the new 'dataset' folder.


In [ ]:
SPLITS = ['train', 'val']

def clean_and_merge_directories(base_path, splits):
    print("Starting Directory Consolidation Protocol...\n")
    
    for split in splits:
        split_dir = os.path.join(base_path, split)
        
        if not os.path.exists(split_dir):
            print(f"Warning: Directory not found: {split_dir}")
            continue
        
        print(f"Scanning {split.upper()} directory: {split_dir}")
        current_folders = [f for f in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, f))]
        
        folders_deleted = 0
        images_moved = 0
        
        for folder in current_folders:
            base_building_name = re.sub(r'[\s_]+[0-9]+$', '', folder).strip()
            if base_building_name == folder:
                continue
                
            source_folder_path = os.path.join(split_dir, folder)
            target_folder_path = os.path.join(split_dir, base_building_name)
            
            os.makedirs(target_folder_path, exist_ok=True)
            
            for filename in os.listdir(source_folder_path):
                source_file = os.path.join(source_folder_path, filename)
                
                if os.path.isfile(source_file):
                    safe_filename = f"{folder}_{filename}"
                    target_file = os.path.join(target_folder_path, safe_filename)
                    
                    shutil.move(source_file, target_file)
                    images_moved += 1
            
            os.rmdir(source_folder_path)
            folders_deleted += 1
            
        print(f"-> Result for {split.upper()}: Moved {images_moved} images and consolidated {folders_deleted} angle-folders.\n")
    print("Data consolidation complete. Your dataset is now ready for production training.")

clean_and_merge_directories(BASE_DIR, SPLITS)

Starting Directory Consolidation Protocol...

Scanning TRAIN directory: C:\Users\Shahbaz\Desktop\dl\dataset\train
-> Result for TRAIN: Moved 1487 images and consolidated 16 angle-folders.

Scanning VAL directory: C:\Users\Shahbaz\Desktop\dl\dataset\val
-> Result for VAL: Moved 380 images and consolidated 16 angle-folders.

Data consolidation complete. Your dataset is now ready for production training.


In [6]:
TRAIN_DIR = os.path.join(BASE_DIR, "dataset", "train")
VAL_DIR = os.path.join(BASE_DIR, "dataset", "val")

def print_folder_stats(directory_path, split_name):
    print(f"\n{'-'*40}")
    print(f"{split_name.upper()} DIRECTORY STATS")
    print(f"{'-'*40}")
    
    if not os.path.exists(directory_path):
        print(f"Directory not found: {directory_path}")
        return 0

    total_images = 0
    # List all the items in the directory
    for folder_name in os.listdir(directory_path):
        folder_path = os.path.join(directory_path, folder_name)
        
        # we are only looking at folders
        if os.path.isdir(folder_path):
            # Count the files inside each folder
            file_count = len(os.listdir(folder_path))
            print(f"{folder_name:<25} : {file_count} images")
            total_images += file_count
            
    print(f"{'-'*40}")
    print(f"TOTAL {split_name.upper()} IMAGES: {total_images}\n")
    return total_images

train_total = print_folder_stats(TRAIN_DIR, "Train")
val_total = print_folder_stats(VAL_DIR, "Validation")


----------------------------------------
TRAIN DIRECTORY STATS
----------------------------------------
basement                  : 161 images
church                    : 1116 images
entrance                  : 217 images
georgianum                : 1020 images
kreuztor                  : 461 images
ku                        : 1569 images
pink                      : 1110 images
room                      : 430 images
wfi                       : 685 images
----------------------------------------
TOTAL TRAIN IMAGES: 6769


----------------------------------------
VALIDATION DIRECTORY STATS
----------------------------------------
basement                  : 41 images
church                    : 280 images
entrance                  : 55 images
georgianum                : 255 images
kreuztor                  : 116 images
ku                        : 393 images
pink                      : 278 images
room                      : 108 images
wfi                       : 172 images
--------------